In [1]:
# Parameters
Month = 2
pao = 1


In [2]:
import os
import io
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# ===== 1. ตั้งค่าพื้นฐาน =====
FOLDER_ID = '1zWlmPBxdypJLydymEmW-9qlDC1LEzLcG'
DOWNLOAD_DIR = 'Supplementary files'
# หาไฟล์ .json ในเครื่องอัตโนมัติ
SERVICE_ACCOUNT_FILE = [f for f in os.listdir('.') if f.endswith('.json')][0]

if not os.path.exists(DOWNLOAD_DIR):
    os.makedirs(DOWNLOAD_DIR)

# ===== 2. เชื่อมต่อ API =====
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
creds = service_account.Credentials.from_service_account_file(
        SERVICE_ACCOUNT_FILE, scopes=SCOPES)
service = build('drive', 'v3', credentials=creds)


def download_folder(folder_id):
    # ค้นหาไฟล์ทั้งหมดที่อยู่ใน Folder นี้ (ไม่เอาไฟล์ในถังขยะ)
    query = f"'{folder_id}' in parents and trashed = false"
    results = service.files().list(q=query, fields="files(id, name, mimeType)").execute()
    items = results.get('files', [])

    if not items:
        print('ไม่พบไฟล์ในโฟลเดอร์นี้')
        return

    print(f"🚀 เริ่มดาวน์โหลดไฟล์ทั้งหมด ({len(items)} ไฟล์)...")

    for item in items:
        file_id = item['id']
        file_name = item['name']
        mime_type = item['mimeType']

        # ถ้าเป็น Google Sheets ต้อง Export เป็น Excel
        if mime_type == 'application/vnd.google-apps.spreadsheet':
            request = service.files().export_media(
                fileId=file_id,
                mimeType='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')
            file_name += '.xlsx'
        else:
            # ถ้าเป็นไฟล์ปกติ (PDF, JPG, XLSX) ให้โหลดตรงๆ
            request = service.files().get_media(fileId=file_id)

        file_path = os.path.join(DOWNLOAD_DIR, file_name)
        
        with io.FileIO(file_path, 'wb') as fh:
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()
            print(f"✅ ดาวน์โหลดเสร็จ: {file_name}")

if __name__ == '__main__':
    download_folder(FOLDER_ID)
    print(f"\n✨ โหลดครบถ้วนที่โฟลเดอร์: {DOWNLOAD_DIR}")

🚀 เริ่มดาวน์โหลดไฟล์ทั้งหมด (3 ไฟล์)...


✅ ดาวน์โหลดเสร็จ: Training_Data.xlsx


✅ ดาวน์โหลดเสร็จ: บันทึกราคาถูกต้อง.xlsx


✅ ดาวน์โหลดเสร็จ: รายการคำนวณและไม่คำนวณทุกชุด.xlsx

✨ โหลดครบถ้วนที่โฟลเดอร์: Supplementary files


In [3]:
# pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib